In [ ]:
# =========================================================
# CELL 1 — Imports + Reproducibility
# Purpose:
#   - Import all dependencies
#   - Fix random seeds for reproducibility
# Inputs:  None
# Outputs: Global seeded RNG state
# =========================================================

import os
import re
import math
import ast
import json
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# NEW CELL 1.5 — Device (define early so later cells can use it)
# =========================================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# =========================================================
# CELL 2 — Load CSVs + Embeddings
# Purpose:
#   - Load all datasets
#   - Load all embedding dicts (id -> np.array)
# Inputs:
#   - Paths to CSVs and .pkl embeddings
# Outputs:
#   - main_df, dialog_df, docs_df, resp_df, summ_df
#   - emb_dialog, emb_docs, emb_resp, emb_summ
# =========================================================

MAIN_PATH   = r"/teamspace/studios/this_studio/datasets/main_mixed_df.csv"
DIALOG_PATH = r"/teamspace/studios/this_studio/datasets/dialogue.csv"
DOCS_PATH   = r"/teamspace/studios/this_studio/datasets/documents.csv"
RESP_PATH   = r"/teamspace/studios/this_studio/datasets/response.csv"
SUMM_PATH   = r"/teamspace/studios/this_studio/datasets/summaries.csv"

EMB_DIALOG_PATH = r"/teamspace/studios/this_studio/embeddings/dialog_embeddings.pkl"
EMB_DOCS_PATH   = r"/teamspace/studios/this_studio/embeddings/doc_embeddings.pkl"
EMB_RESP_PATH   = r"/teamspace/studios/this_studio/embeddings/response_embeddings.pkl"
EMB_SUMM_PATH   = r"/teamspace/studios/this_studio/embeddings/summ_embeddings.pkl"

main_df  = pd.read_csv(MAIN_PATH)
dialog_df= pd.read_csv(DIALOG_PATH)
docs_df  = pd.read_csv(DOCS_PATH)
resp_df  = pd.read_csv(RESP_PATH)
summ_df  = pd.read_csv(SUMM_PATH)

def load_embeddings(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

emb_dialog = load_embeddings(EMB_DIALOG_PATH)
emb_docs   = load_embeddings(EMB_DOCS_PATH)
emb_resp   = load_embeddings(EMB_RESP_PATH)
emb_summ   = load_embeddings(EMB_SUMM_PATH)

print("Loaded main_df columns:", list(main_df.columns))
print("Rows:", len(main_df))
print("Sample row:\n", main_df.iloc[0])

In [3]:
# =========================================================
# CELL 3 — Core Helpers (sequence parse, canonical ids, null checks)
# Purpose:
#   - Robust parsing of serialized sequences
#   - Canonical ID utilities
# Inputs:  main_df cell strings
# Outputs: Helper functions used everywhere
# =========================================================

CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def is_nullish(x):
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "null", "none", "nan"}

def normalize_id(x):
    return str(x).strip()

def to_canonical(prefix: str, raw_id):
    s = str(raw_id).strip()
    if CANON_RE.match(s):
        return s
    return f"{prefix}-{s}"

def parse_serialized_sequence(x):
    """
    Accepts:
      - Python list
      - '["a","b"]'
      - '[a,b]'
    Returns: list[str]
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip() != ""]
    s = str(x).strip()
    if not (s.startswith("[") and s.endswith("]")):
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(t).strip() for t in v if str(t).strip() != ""]
    except Exception:
        pass
    items = [t.strip().strip('"').strip("'") for t in s[1:-1].split(",")]
    return [t for t in items if t]

def fix_and_align_sequences(nodes, actions, max_len=100):
    nodes = list(nodes)
    actions = list(actions)
    L = min(len(nodes), len(actions), max_len)
    return nodes[:L], actions[:L]

In [ ]:
# =========================================================
# CELL 4 — Build id2text (canonical) from all text-bearing tables
# Purpose:
#   - Create mapping canonical_id -> text
# Inputs:
#   - dialog_df, docs_df, resp_df, summ_df
# Outputs:
#   - id2text dict
# =========================================================

def build_id2text(dialog_df, docs_df, resp_df, summ_df):
    id2text = {}

    # Dialogue: dl-*
    if "dialogue_id" in dialog_df.columns:
        for _, r in dialog_df.dropna(subset=["dialogue_id"]).iterrows():
            cid = to_canonical("dl", r["dialogue_id"])
            id2text[cid] = str(r.get("dialogue_text", "")).strip()

    # Docs: d-*
    if "NewsID" in docs_df.columns:
        for _, r in docs_df.dropna(subset=["NewsID"]).iterrows():
            cid = to_canonical("d", r["NewsID"])
            nb = str(r.get("NewsBody", "")).strip()
            hl = str(r.get("Headline", "")).strip()
            id2text[cid] = nb if nb else hl

    # Responses: r-*
    if "response_id" in resp_df.columns:
        for _, r in resp_df.dropna(subset=["response_id"]).iterrows():
            cid = to_canonical("r", r["response_id"])
            id2text[cid] = str(r.get("response_text", "")).strip()

    # Summaries: s-*
    if "SummID" in summ_df.columns:
        for _, r in summ_df.dropna(subset=["SummID"]).iterrows():
            cid = to_canonical("s", r["SummID"])
            id2text[cid] = str(r.get("Summary", "")).strip()

    return id2text

id2text = build_id2text(dialog_df, docs_df, resp_df, summ_df)
print("id2text size:", len(id2text))
print("sample keys:", list(id2text.keys())[:10])


In [ ]:
# =========================================================
# CELL 5 — Embedding Resolver for canonical ids
# Purpose:
#   - Provide emb_resolver.get(cid) -> np.array(768,)
# Inputs:
#   - emb_dialog, emb_docs, emb_resp, emb_summ
# Outputs:
#   - emb_resolver
# =========================================================

class EmbLookup:
    @staticmethod
    def lookup(emb_dict, key):
        if key in emb_dict:
            return emb_dict[key]
        s = str(key)
        if s in emb_dict:
            return emb_dict[s]
        nums = re.findall(r"\d+", s)
        if nums:
            n = nums[-1]
            if n in emb_dict:
                return emb_dict[n]
            try:
                ni = int(n)
                if ni in emb_dict:
                    return emb_dict[ni]
            except Exception:
                pass
        return None

class CanonicalEmbeddingResolver:
    def __init__(self, emb_dialog, emb_docs, emb_resp, emb_summ, d=768):
        self.emb_dialog = emb_dialog
        self.emb_docs = emb_docs
        self.emb_resp = emb_resp
        self.emb_summ = emb_summ
        self.d = d

    def get(self, cid: str) -> np.ndarray:
        cid = normalize_id(cid)
        prefix = cid.split("-")[0].lower()

        if prefix == "dl":
            v = EmbLookup.lookup(self.emb_dialog, cid)
        elif prefix == "d":
            v = EmbLookup.lookup(self.emb_docs, cid)
        elif prefix == "r":
            v = EmbLookup.lookup(self.emb_resp, cid)
        elif prefix == "s":
            v = EmbLookup.lookup(self.emb_summ, cid)
        else:
            v = None

        if v is None:
            return np.zeros((self.d,), dtype=np.float32)

        v = np.asarray(v, dtype=np.float32)
        if v.shape[-1] != self.d:
            raise ValueError(f"Embedding dim mismatch for {cid}: got {v.shape}, expected ({self.d},)")
        return v

emb_resolver = CanonicalEmbeddingResolver(emb_dialog, emb_docs, emb_resp, emb_summ, d=768)
print("Embedding resolver ready.")


In [ ]:
# =========================================================
# CELL 6 — Task Names + Task IDs
# Purpose:
#   - Build task vocab for task-classification head (all tasks)
# Inputs:
#   - main_df['task']
# Outputs:
#   - task2idx, idx2task, main_df['task_id'], n_tasks
# =========================================================

assert "task" in main_df.columns, "main_df must contain 'task'"

task_names = sorted(main_df["task"].dropna().astype(str).unique())
task2idx = {t: i for i, t in enumerate(task_names)}
idx2task = {i: t for t, i in task2idx.items()}

main_df["task_id"] = main_df["task"].astype(str).map(task2idx)
assert main_df["task_id"].isna().sum() == 0, "Unmapped task found"

n_tasks = len(task2idx)
print("Tasks:", task2idx)


In [ ]:
# =========================================================
# CELL 7 — Build ACTION vocab (all rows) and NODE vocab (all rows)
# Purpose:
#   - actions2idx for sequence action loss
#   - node2idx for sequence node loss
# Inputs:
#   - main_df['interactions'], main_df['text_nodes']
# Outputs:
#   - actions2idx/idx2action
#   - node2idx/idx2node
# =========================================================

# ACTIONS
all_actions = set()
for seq in main_df["interactions"].dropna():
    all_actions.update(parse_serialized_sequence(seq))

actions2idx = {a: i for i, a in enumerate(sorted(all_actions))}
idx2action  = {i: a for a, i in actions2idx.items()}

# NODES (sequence tokens)
node_ids = set()
for seq in main_df["text_nodes"].dropna():
    for nid in parse_serialized_sequence(seq):
        if not is_nullish(nid):
            node_ids.add(normalize_id(nid))

# Include query/target ids if present (safer for lookups)
for col in ["query_text_id", "target_text_id"]:
    if col in main_df.columns:
        for x in main_df[col].dropna():
            if not is_nullish(x):
                node_ids.add(normalize_id(x))

node2idx = {nid: i for i, nid in enumerate(sorted(node_ids))}
idx2node = {i: nid for nid, i in node2idx.items()}

print("VOCAB SIZES:")
print("  actions:", len(actions2idx))
print("  nodes  :", len(node2idx))


In [ ]:
# =========================================================
# CELL 8 — Build USER vocab ONLY from next_user_prediction rows
# Purpose:
#   - user vocabulary restricted to next_user_prediction task
# Inputs:
#   - main_df rows where task == 'next_user_prediction'
# Outputs:
#   - user2idx_small / idx2user_small
# Notes:
#   - For non-user-pred tasks, user loss will be masked out
# =========================================================

USER_TASK = "next_user_prediction"

user_rows = main_df[main_df["task"].astype(str).str.strip() == USER_TASK]
unique_users_small = sorted(user_rows["userid"].dropna().astype(str).unique())

# ensure at least an <unk> for safety
unique_users_small = ["<unk_user>"] + [u for u in unique_users_small if u != "<unk_user>"]

user2idx_small = {u: i for i, u in enumerate(unique_users_small)}
idx2user_small = {i: u for u, i in user2idx_small.items()}

print("User vocab size (next_user_prediction only):", len(user2idx_small))


In [ ]:
# =========================================================
# CELL 9 — Build ITEM vocab ONLY from next_item_prediction rows (query_text_id only)
# Purpose:
#   - item vocabulary restricted to next_item_prediction task
# Inputs:
#   - main_df rows where task == 'next_item_prediction'
# Outputs:
#   - item2idx / idx2item
# Notes:
#   - Vocab is built ONLY from query_text_id by your requirement
#   - Targets (target_text_id) might be OOV; handled via <unk_item>
# =========================================================

ITEM_TASK = "next_item_prediction"

item_rows = main_df[main_df["task"].astype(str).str.strip() == ITEM_TASK]
q_items = item_rows["query_text_id"].dropna().astype(str).map(lambda x: normalize_id(x)).unique().tolist()

q_items = ["<unk_item>"] + [x for x in sorted(set(q_items)) if x != "<unk_item>"]

item2idx = {it: i for i, it in enumerate(q_items)}
idx2item = {i: it for it, i in item2idx.items()}

print("Item vocab size (next_item_prediction query_text_id only):", len(item2idx))


In [ ]:
# =========================================================
# CELL 10 — Build word supervision vocab ONLY for dialogue_generation + personalized_summarization
# Purpose:
#   - Replace YAKE phrases with simple word lists
#   - Build id2words ONLY using query_text_id + target_text_id columns
#     and ONLY for tasks: dialogue_generation, personalized_summarization
# Inputs:
#   - main_df filtered tasks
#   - id2text
# Outputs:
#   - id2words: canonical_id -> list[str]
#   - word2idx/idx2word
# =========================================================

GEN_TASKS_WORD = {"dialogue_generation", "personalized_summarization"}

WORD_MAX_PER_TEXT = 80
WORD_MIN_LEN = 2

WORD_RE = re.compile(r"[a-z0-9]+")  # simple tokenization

def tokenize_words(text: str):
    text = (text or "").lower()
    toks = WORD_RE.findall(text)
    toks = [t for t in toks if len(t) >= WORD_MIN_LEN]
    return toks

# collect unique ids only from query/target cols for these tasks
gen_df = main_df[main_df["task"].astype(str).str.strip().isin(GEN_TASKS_WORD)].copy()

need_cols = ["query_text_id", "target_text_id"]
for c in need_cols:
    assert c in gen_df.columns, f"Missing column {c} for word supervision."

unique_text_ids = set()
for c in need_cols:
    unique_text_ids.update(gen_df[c].dropna().astype(str).map(normalize_id).tolist())

# id2words (only for ids we can resolve to text)
id2words = {}
word_vocab = set()

for tid in unique_text_ids:
    if tid not in id2text:
        continue
    words = tokenize_words(id2text[tid])[:WORD_MAX_PER_TEXT]
    id2words[tid] = words
    word_vocab.update(words)

# vocab with <pad>/<unk> for safety (multi-label does not need pad, but keep unk)
word_vocab = ["<unk_word>"] + [w for w in sorted(word_vocab) if w != "<unk_word>"]
word2idx = {w: i for i, w in enumerate(word_vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print("Texts for word supervision:", len(id2words))
print("Word vocab size:", len(word2idx))

In [ ]:
# =========================================================
# Build per-word importance weights (with progress bar)
# =========================================================

import spacy
from tqdm import tqdm

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # faster
CONTENT_POS = {"NOUN", "PROPN", "ADJ"}

word_importance = torch.ones(len(word2idx), dtype=torch.float32)

# We iterate in vocab order for stability
words = list(word2idx.keys())

for w in tqdm(words, desc="Computing word importance"):
    i = word2idx[w]
    doc = nlp(w)
    if len(doc) > 0 and doc[0].pos_ in CONTENT_POS:
        word_importance[i] = 10.0     # important content words
    else:
        word_importance[i] = 0.02     # function / glue words

word_importance = word_importance.to(device)

print("Word importance ready:", word_importance.shape)


In [ ]:
# =========================================================
# CELL 11 — Split train/test from provided 'split' column (no dropping)
# Purpose:
#   - Use main_df['split'] == train/test
# Inputs:
#   - main_df['split']
# Outputs:
#   - train_df, test_df
# =========================================================

assert "split" in main_df.columns, "main_df must contain 'split' with train/test."

main_df["split"] = main_df["split"].astype(str).str.strip().str.lower()
train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df), "| Test size:", len(test_df))
print("Task counts (train):\n", train_df["task"].value_counts())
print("Task counts (test):\n", test_df["task"].value_counts())

In [13]:
# =========================================================
# CELL 12 — Batch Builder (row -> tensors) with TASK-WISE masks
# (DROP-IN REPLACEMENT: adds q_text/t_text for LM)
# =========================================================

MAX_T = 60
D_MODEL = 768
IGNORE_INDEX = -100

def build_multi_hot_words(text_id: str, device):
    """
    Multi-hot vector over word2idx for the given canonical text_id.
    """
    y = torch.zeros(len(word2idx), device=device)
    words = id2words.get(text_id, [])
    for w in words:
        j = word2idx[w] if w in word2idx else 0
        y[j] = 1.0
    return y, words

def safe_user_label(userid: str):
    userid = str(userid).strip()
    if userid in user2idx_small:
        return user2idx_small[userid]
    return 0

def safe_item_label(itemid: str):
    itemid = normalize_id(itemid)
    if itemid in item2idx:
        return item2idx[itemid]
    return 0

def build_row_tensors(row, device):
    task = str(row["task"]).strip()
    task_id = int(row["task_id"])

    # ---- sequences ----
    actions = parse_serialized_sequence(row["interactions"])
    nodes   = parse_serialized_sequence(row["text_nodes"])

    if len(actions) == 0 or len(nodes) == 0:
        return {"ok": False, "reason": "empty_sequence", "task": task}

    nodes, actions = fix_and_align_sequences(nodes, actions, max_len=MAX_T)
    if len(nodes) < 3:
        return {"ok": False, "reason": "too_short_sequence", "task": task}

    T = len(nodes)

    # ---- action tensors ----
    try:
        action_ids = [actions2idx[a] for a in actions]
    except KeyError:
        return {"ok": False, "reason": "unknown_action_token", "task": task}

    action_labels_full = torch.tensor(action_ids, device=device).unsqueeze(0)  # (1,T)
    action_onehot = F.one_hot(action_labels_full, num_classes=len(actions2idx)).float()  # (1,T,A)

    # ---- node tensors ----
    nodes_norm = [normalize_id(n) for n in nodes]
    try:
        node_ids = [node2idx[n] for n in nodes_norm]
    except KeyError:
        return {"ok": False, "reason": "unknown_node_token", "task": task}

    node_labels_full = torch.tensor(node_ids, device=device).unsqueeze(0)  # (1,T)

    # ---- embeddings sequence ----
    e_seq = torch.tensor(
        np.stack([emb_resolver.get(n) for n in nodes_norm], axis=0),
        device=device
    ).unsqueeze(0)  # (1,T,768)

    # ---- query/target ids ----
    qid = normalize_id(row["query_text_id"]) if "query_text_id" in row else nodes_norm[-2]
    tid = normalize_id(row["target_text_id"]) if "target_text_id" in row else nodes_norm[-1]

    q_text_emb = torch.tensor(emb_resolver.get(qid), device=device).unsqueeze(0)  # (1,768)

    # ---- resolve raw texts (for frozen LM prompt/labels) ----
    q_text = str(id2text.get(qid)).strip()
    t_text = str(id2text.get(tid)).strip()

    # ---- task label ----
    task_labels = torch.tensor([task_id], device=device)

    # ---- sequence loss masking rule ----
    if task == ITEM_TASK:
        keep_len = max(1, T - 2)
    else:
        keep_len = T

    action_labels = action_labels_full.clone()
    node_labels   = node_labels_full.clone()

    if keep_len < T:
        action_labels[0, keep_len:] = IGNORE_INDEX
        node_labels[0, keep_len:]   = IGNORE_INDEX

    # ---- user prediction target/mask ----
    user_mask = torch.tensor([1.0 if task == USER_TASK else 0.0], device=device)
    user_labels = torch.tensor([safe_user_label(row["userid"])], device=device)

    # ---- item prediction target/mask ----
    item_mask = torch.tensor([1.0 if task == ITEM_TASK else 0.0], device=device)
    item_target_id = safe_item_label(tid)
    item_labels = torch.tensor([item_target_id], device=device)

    # ---- word decoder supervision/mask ----
    word_mask = torch.tensor([1.0 if task in GEN_TASKS_WORD else 0.0], device=device)

    word_targets = torch.zeros(1, len(word2idx), device=device)
    gold_words = []
    if task in GEN_TASKS_WORD:
        y, gold_words = build_multi_hot_words(tid, device=device)
        word_targets[0] = y

    return {
        "ok": True,
        "task": task,
        "T": T,

        "e_seq": e_seq,
        "action_onehot": action_onehot,
        "action_labels": action_labels,
        "node_labels": node_labels,

        "task_labels": task_labels,
        "q_text_emb": q_text_emb,

        "user_labels": user_labels,
        "user_mask": user_mask,

        "item_labels": item_labels,
        "item_mask": item_mask,

        "word_targets": word_targets,
        "word_mask": word_mask,
        "gold_words": gold_words,

        "qid": qid,
        "tid": tid,
        "q_text": q_text,
        "t_text": t_text,

        "userid_raw": str(row["userid"]).strip(),
    }


In [14]:
# =========================================================
# CELL 12.9 (NEW) — Loss Utilities (Focal Cross-Entropy with reduction)
# Purpose:
#   - Cross-entropy variant that increases weight on incorrect predictions
#   - Supports reduction='none' for masking
# Inputs:
#   - logits: (B,C)
#   - targets: (B,)
# Outputs:
#   - loss: (B,) if reduction='none' else scalar
# =========================================================

class FocalCELoss(nn.Module):
    def __init__(self, gamma=2.0, reduction="mean"):
        super().__init__()
        assert reduction in {"none", "mean", "sum"}
        self.gamma = float(gamma)
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B,C), targets: (B,)
        logp = F.log_softmax(logits, dim=-1)                  # (B,C)
        p = torch.exp(logp)                                   # (B,C)
        pt = p.gather(1, targets.unsqueeze(1)).squeeze(1)      # (B,)
        logpt = logp.gather(1, targets.unsqueeze(1)).squeeze(1)# (B,)
        loss = -((1.0 - pt) ** self.gamma) * logpt            # (B,)

        if self.reduction == "none":
            return loss
        if self.reduction == "sum":
            return loss.sum()
        return loss.mean()


In [15]:
# =========================================================
# CELL 13 (REPLACE) — UniHis SSM Encoder (same architecture, task loss changed)
# Purpose:
#   - SSM history encoder with task prediction head
#   - Shared sequence heads for action/node (masked via IGNORE_INDEX)
# Inputs:
#   - e_seq, action_onehot, action_labels, node_labels, task_labels, q_text_emb
# Outputs:
#   - losses: loss_action, loss_node, loss_task (FOCAL-CE)
#   - states: h_final (task-conditioned), P_task, logits
# =========================================================

class UniHisSSMEncoder(nn.Module):
    def __init__(
        self,
        vocab_sizes,
        d_model=768,
        d_state=64,
        n_tasks=8,
        d_qtext=768,
        known_task=False
    ):
        super().__init__()
        self.d_state = d_state
        self.known_task = known_task

        # sequence CE (unchanged)
        self.criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

        # task CE (changed to focal CE)
        self.task_criterion = FocalCELoss(gamma=2.0, reduction="mean")

        # ========= SSM EIGEN PARAMS =========
        def init_eig():
            Q = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
            Lambda = nn.Parameter(torch.empty(d_state).uniform_(-1.0, -0.1))
            return Q, Lambda

        self.Q_L, self.Lambda_L = init_eig()
        self.Q_S, self.Lambda_S = init_eig()
        self.Q_E, self.Lambda_E = init_eig()

        self.B = nn.Parameter(torch.randn(d_state, d_model))

        self.tau_S = nn.Parameter(torch.tensor(1.0))
        self.tau_E = nn.Parameter(torch.tensor(1.0))

        # ========= ACTION → MOOD =========
        self.action_proj = nn.Linear(vocab_sizes["actions"], d_model)

        self.MLP_mood = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        self.W_a = nn.Linear(d_model, d_state)

        # ========= MIXERS =========
        self.Mixer_L = nn.Linear(d_state, d_state)
        self.Mixer_S = nn.Linear(d_state, d_state)
        self.Mixer_E = nn.Linear(d_state, d_state)

        self.W_fuse = nn.Linear(3 * d_state, 3)
        self.Mixer_Fuse = nn.Linear(d_state, d_state)

        # ========= HEADS =========
        self.action_head = nn.Linear(3 * d_state, vocab_sizes["actions"])
        self.node_head   = nn.Linear(3 * d_state, vocab_sizes["nodes"])

        # ========= TASK =========
        self.Embed_Q = nn.Linear(d_qtext, d_state)
        self.W_task  = nn.Linear(2 * d_state, n_tasks)

        self.W_hyp = nn.Linear(2 * d_state, d_state)
        self.W_theta = nn.ModuleList([
            nn.Linear(d_state, 2 * d_state) for _ in range(n_tasks)
        ])

    # ========= SSM MATRIX =========
    def make_A(self, Q, Lambda_diag):
        Q_inv = torch.linalg.inv(Q)
        return Q @ torch.diag(Lambda_diag) @ Q_inv

    # ========= FORWARD =========
    def forward(
        self,
        e_seq,             # (B, T, d_model)
        action_onehot,     # (B, T, |A|)
        action_labels,     # (B, T) with IGNORE_INDEX for masked steps
        node_labels,       # (B, T) with IGNORE_INDEX for masked steps
        task_labels,       # (B,)
        q_text_emb         # (B, d_qtext)
    ):
        B, T, _ = e_seq.shape
        device = e_seq.device

        # ========= INITIAL STATES =========
        xL = torch.zeros(B, self.d_state, device=device)
        xS = torch.zeros(B, self.d_state, device=device)
        xE = torch.zeros(B, self.d_state, device=device)

        act_logits = []
        node_logits = []

        # ========= SSM MATRICES =========
        A_L = self.make_A(self.Q_L, torch.exp(self.Lambda_L))

        eps_S = torch.sigmoid(self.tau_S)
        k = torch.log((1 - eps_S) / eps_S) / torch.abs(self.Lambda_S)
        A_S = self.make_A(self.Q_S, torch.sigmoid(-k * self.Lambda_S))

        eta = (torch.exp(-1.0 / self.tau_E) - 1.0) / self.Lambda_E
        A_E = self.make_A(self.Q_E, 1.0 + eta * self.Lambda_E)

        # ========= SEQUENCE LOOP =========
        for t in range(T):
            e_t = e_seq[:, t]                      # (B, d_model)
            a_t = action_onehot[:, t, :]           # (B, |A|)

            a_emb = self.action_proj(a_t)
            mood_vec = self.MLP_mood(a_emb)
            mood = torch.sigmoid(self.W_a(mood_vec))  # (B,d_state)

            u_t = torch.bmm(
                (self.B.unsqueeze(0) * mood.unsqueeze(2)),
                e_t.unsqueeze(2)
            ).squeeze(2)  # (B,d_state)

            xL = torch.matmul(xL, A_L.T) + u_t
            xS = torch.matmul(xS, A_S.T) + u_t
            xE = torch.matmul(xE, A_E.T) + u_t

            x_cat = torch.cat([xL, xS, xE], dim=-1)  # (B,3d_state)

            act_logits.append(self.action_head(x_cat))
            node_logits.append(self.node_head(x_cat))

        act_logits  = torch.stack(act_logits, dim=1)     # (B, T, |A|)
        node_logits = torch.stack(node_logits, dim=1)    # (B, T, |V|)

        # ========= HISTORY FUSION =========
        yL = self.Mixer_L(xL)
        yS = self.Mixer_S(xS)
        yE = self.Mixer_E(xE)

        w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)
        h_hist = self.Mixer_Fuse(
            w[:, 0:1] * yL +
            w[:, 1:2] * yS +
            w[:, 2:3] * yE
        )

        # ========= ACTION + NODE LOSS (unchanged) =========
        loss_action = self.criterion(
            act_logits.view(-1, act_logits.size(-1)),
            action_labels.view(-1)
        )

        loss_node = self.criterion(
            node_logits.view(-1, node_logits.size(-1)),
            node_labels.view(-1)
        )

        # ========= TASK CONDITIONING =========
        q_emb = self.Embed_Q(q_text_emb)
        z_switch = torch.cat([h_hist, q_emb], dim=-1)

        task_logits = self.W_task(z_switch)
        P_task = F.softmax(task_logits, dim=-1)

        # TASK LOSS (changed)
        loss_task = self.task_criterion(task_logits, task_labels)

        h_hyp = F.relu(self.W_hyp(z_switch))

        if self.known_task:
            theta = self.W_theta[task_labels[0]](h_hyp)
            g, b = theta.chunk(2, dim=-1)
            h_final = torch.sigmoid(g) * h_hist + b
        else:
            h_final = 0.0
            for i in range(len(self.W_theta)):
                theta_i = self.W_theta[i](h_hyp)
                g_i, b_i = theta_i.chunk(2, dim=-1)
                h_final = h_final + P_task[:, i:i+1] * (
                    torch.sigmoid(g_i) * h_hist + b_i
                )

        return {
            "loss_action": loss_action,
            "loss_node": loss_node,
            "loss_task": loss_task,

            "act_logits": act_logits,
            "node_logits": node_logits,

            "task_logits": task_logits,
            "P_task": P_task,

            "h_final": h_final
        }


In [16]:
# =========================================================
# CELL 14 — Task Heads:
#   (A) User MLP head (only for next_user_prediction)
#   (B) Item MLP head (only for next_item_prediction)
# Purpose:
#   - Implement your requested small MLP classifiers
# Inputs:
#   - h_final
# Outputs:
#   - user_logits, item_logits
# =========================================================

class MLPClassifier(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_out)
        )

    def forward(self, x):
        return self.net(x)

class TaskHeads(nn.Module):
    def __init__(self, d_state, n_users_small, n_items, dropout=0.1):
        super().__init__()
        # "proper small mlp" for users
        self.user_head = MLPClassifier(d_state, d_state // 2, n_users_small, dropout=dropout)
        # 2-layer MLP for next-item over item vocab
        self.item_head = MLPClassifier(d_state, d_state, n_items, dropout=dropout)

    def forward(self, h_final):
        return {
            "user_logits": self.user_head(h_final),
            "item_logits": self.item_head(h_final),
        }


In [17]:
# =========================================================
# CELL 15 (REPLACE) — RWKV-style Word Decoder (asymmetric BCE: false positives expensive)
# Purpose:
#   - Word multi-label decoder used only for dialogue_generation + personalized_summarization
#   - Penalize predicting wrong words heavily (high false-positive cost)
#   - Penalize missing gold words lightly (low false-negative cost)
# Inputs:
#   - h_final, q_text_emb, P_task, word_targets, word_mask
# Outputs:
#   - word_logits, loss_word (masked)
# =========================================================

class LensGate(nn.Module):
    def __init__(self, d_state, d_qtext):
        super().__init__()
        self.Wq = nn.Linear(d_qtext, d_state)
        self.Wfilm = nn.Linear(d_state, 2 * d_state)
        self.Wg = nn.Linear(2 * d_state, d_state)

    def forward(self, h_final, q_text_emb):
        q0 = self.Wq(q_text_emb)
        gamma, beta = self.Wfilm(h_final).chunk(2, dim=-1)
        q_lens = (1.0 + gamma) * q0 + beta
        g = torch.sigmoid(self.Wg(torch.cat([h_final, q_lens], dim=-1)))
        return g * q_lens + (1.0 - g) * q0


class RWKVTimeMix(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, d, bias=False)
        self.value = nn.Linear(d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.output = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = self.key(xm)
        v = self.value(xm)

        w = torch.softmax(k, dim=1)
        mix = torch.sum(w * v, dim=1, keepdim=True)
        y = self.output(r * mix).expand_as(x)
        return x + y


class RWKVChannelMix(nn.Module):
    def __init__(self, d, expansion=4):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, expansion * d, bias=False)
        self.value = nn.Linear(expansion * d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = torch.relu(self.key(xm))
        v = self.value(k)
        return x + r * v


class RWKVWordDecoder(nn.Module):
    def __init__(self, d_state, d_qtext, n_tasks, n_words, n_layers=2,
                 neg_weight=10.0, pos_weight=1.0):
        super().__init__()

        # Asymmetry:
        #   - neg_weight: penalty for false positives (predicting wrong)
        #   - pos_weight: penalty for false negatives (missing gold)
        self.neg_weight = float(neg_weight)
        self.pos_weight = float(pos_weight)

        self.lens_gate = LensGate(d_state, d_qtext)

        self.task_embed = nn.Embedding(n_tasks, d_state)
        self.task_proj = nn.Linear(d_state, d_state)

        self.proj_h = nn.Linear(d_state, d_state)
        self.proj_q = nn.Linear(d_state, d_state)

        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(RWKVTimeMix(d_state))
            self.blocks.append(RWKVChannelMix(d_state))

        self.out_ln = nn.LayerNorm(d_state)
        self.word_head = nn.Linear(d_state, n_words)

    def forward(self, h_final, q_text_emb, P_task, word_targets, word_mask):
        # h_final: (B,d_state)
        # q_text_emb: (B,d_qtext)
        # P_task: (B,n_tasks)
        # word_targets: (B,n_words) multi-hot
        # word_mask: (B,) 1 for gen tasks else 0

        B = h_final.size(0)

        q_cond = self.lens_gate(h_final, q_text_emb)

        task_ids = torch.arange(P_task.size(1), device=h_final.device)
        Temb = self.task_embed(task_ids)                 # (n_tasks,d)
        prompt = self.task_proj(P_task @ Temb)           # (B,d)

        token_h = self.proj_h(h_final + prompt)          # (B,d)
        token_q = self.proj_q(q_cond + prompt)           # (B,d)

        x = torch.stack([token_h, token_q], dim=1)       # (B,2,d)

        for blk in self.blocks:
            x = blk(x)

        h_dec = self.out_ln(x[:, -1])                    # (B,d)
        word_logits = self.word_head(h_dec)              # (B,n_words)

        # ===== Asymmetric BCE (false positives expensive) =====
        # probabilities
        p = torch.sigmoid(word_logits)

        # False positive: target=0 but predicted high -> must be VERY expensive
        fp = -(1.0 - word_targets) * torch.log(1.0 - p + 1e-12)

        # False negative: target=1 but predicted low -> cheap
        fn = -word_targets * torch.log(p + 1e-12)

        # Strong precision bias: fp >> fn
        NEG_WEIGHT = 20.0   # punish hallucinated words
        POS_WEIGHT = 1.0    # tolerate missing gold words

        # apply per-word importance weights (same vocab, same shape)
        # word_importance shape: (n_words,)
        weighted_loss = word_importance.unsqueeze(0) * (NEG_WEIGHT * fp + POS_WEIGHT * fn)

        loss_word_per = weighted_loss.mean(dim=-1)


        # mask non-generation tasks
        loss_word = (loss_word_per * word_mask).sum() / (word_mask.sum() + 1e-12)


        return {
            "word_logits": word_logits,
            "loss_word": loss_word
        }


In [18]:
# =========================================================
# CELL 16 (REPLACE) — Full Multi-Task Model Wrapper + Frozen-LM loss for GEN tasks
# =========================================================

class UniHisMultiTaskModel(nn.Module):
    def __init__(
        self,
        encoder: UniHisSSMEncoder,
        d_qtext: int,
        n_tasks: int,
        n_users_small: int,
        n_items: int,
        n_words: int,
        rwkv_layers: int = 2
    ):
        super().__init__()
        self.encoder = encoder
        self.heads = TaskHeads(d_state=encoder.d_state, n_users_small=n_users_small, n_items=n_items)

        self.word_decoder = RWKVWordDecoder(
            d_state=encoder.d_state,
            d_qtext=d_qtext,
            n_tasks=n_tasks,
            n_words=n_words,
            n_layers=rwkv_layers,
            neg_weight=10.0,
            pos_weight=1.0
        )

        self.user_criterion = FocalCELoss(gamma=2.0, reduction="none")
        self.item_criterion = FocalCELoss(gamma=2.0, reduction="none")

        # ---- Frozen LM is attached later ----
        self.lm = None
        self.lm_tokenizer = None

        # adapter params (created when LM is set)
        self.lm_word_soft_emb = None   # (V_words, d_bridge)
        self.lm_hint_proj = None       # (d_bridge -> hint_len*d_lm)

        # settings
        self.lm_hint_len = 0
        self.lm_d_bridge = 0
        self.lm_d_model = 0
        self.lm_word_temp = 1.0
        self.lm_loss_weight = 1.0
        self.lm_max_prompt_tokens = 256
        self.lm_max_target_tokens = 256

    # ---------- parameter routing for optimizer compatibility ----------
    def base_param_list(self):
        # everything except lm_* params
        return [p for n, p in self.named_parameters() if not n.startswith("lm_")]

    def lm_param_list(self):
        # only lm_* params
        return [p for n, p in self.named_parameters() if n.startswith("lm_")]

    # ---------- attach frozen LM + build differentiable bridge ----------
    def set_frozen_lm(
        self,
        lm_model,
        lm_tokenizer,
        hint_len: int = 8,
        d_bridge: int = 128,
        word_temp: float = 1.0,
        loss_weight: float = 1.0,
        max_prompt_tokens: int = 256,
        max_target_tokens: int = 256,
    ):
        self.lm = lm_model
        self.lm_tokenizer = lm_tokenizer

        self.lm_hint_len = int(hint_len)
        self.lm_d_bridge = int(d_bridge)
        self.lm_d_model = int(self.lm.get_input_embeddings().embedding_dim)

        self.lm_word_temp = float(word_temp)
        self.lm_loss_weight = float(loss_weight)

        self.lm_max_prompt_tokens = int(max_prompt_tokens)
        self.lm_max_target_tokens = int(max_target_tokens)

        # Create differentiable mapping: RWKV word logits -> soft hint embeddings (hint_len tokens)
        Vw = self.word_decoder.word_head.out_features  # n_words
        self.lm_word_soft_emb = nn.Parameter(torch.randn(Vw, self.lm_d_bridge) * 0.02)
        self.lm_hint_proj = nn.Linear(self.lm_d_bridge, self.lm_hint_len * self.lm_d_model)

    def _tokenize_trunc(self, text: str, max_len: int, device):
        enc = self.lm_tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=max_len
        )
        return enc["input_ids"].to(device)

    def _build_prompt(self, q_text: str):
        # text-only prompt as requested
        return (
            "Context:\n"
            + (q_text or "")
            + "\n\n"
            "Instruction: Write the best possible response.\n"
            "Response:\n"
        )

    def _frozen_lm_loss(self, word_logits, q_text: str, t_text: str, device):
        """
        Differentiable LM loss:
          - prompt is text tokens
          - RWKV word logits produce soft hint embeddings inserted after prompt
          - gold target text is teacher-forced, loss computed on target tokens only
        """
        if self.lm is None or self.lm_tokenizer is None:
            return None

        q_text = (q_text or "").strip()
        t_text = (t_text or "").strip()
        if len(q_text) == 0 or len(t_text) == 0:
            return None

        prompt = self._build_prompt(q_text)

        prompt_ids = self._tokenize_trunc(prompt, self.lm_max_prompt_tokens, device)
        target_ids = self._tokenize_trunc(t_text, self.lm_max_target_tokens, device)

        if target_ids.numel() == 0:
            return None

        # embeddings
        E = self.lm.get_input_embeddings()
        prompt_embeds = E(prompt_ids)   # (1, Lp, d_lm)
        target_embeds = E(target_ids)   # (1, Lt, d_lm)

        # RWKV -> soft word distribution -> bridge embedding
        # word_logits: (B,Vw); here B=1 in your pipeline
        p_words = torch.softmax(word_logits / self.lm_word_temp, dim=-1)  # (1,Vw)
        e_bridge = p_words @ self.lm_word_soft_emb                          # (1,d_bridge)

        hint_flat = self.lm_hint_proj(e_bridge)                             # (1, hint_len*d_lm)
        hint_embeds = hint_flat.view(1, self.lm_hint_len, self.lm_d_model)  # (1,hint_len,d_lm)

        # full inputs: prompt + hint + gold target
        inputs_embeds = torch.cat([prompt_embeds, hint_embeds, target_embeds], dim=1)

        Lp = prompt_embeds.size(1)
        Lh = hint_embeds.size(1)
        Lt = target_embeds.size(1)
        L = Lp + Lh + Lt

        attention_mask = torch.ones((1, L), device=device, dtype=torch.long)

        # labels: ignore prompt + hint, supervise target only
        labels = torch.full((1, L), -100, device=device, dtype=torch.long)
        labels[:, Lp + Lh:] = target_ids

        out = self.lm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False
        )
        return out.loss  # scalar

    def forward(self, batch):
        enc = self.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h_final = enc["h_final"]
        head_out = self.heads(h_final)

        # ---------- user loss (masked) ----------
        user_logits = head_out["user_logits"]
        user_labels = batch["user_labels"]
        user_mask   = batch["user_mask"]
        loss_user_per = self.user_criterion(user_logits, user_labels)
        loss_user = (loss_user_per * user_mask).sum() / (user_mask.sum() + 1e-8)

        # ---------- item loss (masked) ----------
        item_logits = head_out["item_logits"]
        item_labels = batch["item_labels"]
        item_mask   = batch["item_mask"]
        loss_item_per = self.item_criterion(item_logits, item_labels)
        loss_item = (loss_item_per * item_mask).sum() / (item_mask.sum() + 1e-8)

        # ---------- word loss ----------
        dec = self.word_decoder(
            h_final=enc["h_final"],
            q_text_emb=batch["q_text_emb"],
            P_task=enc["P_task"],
            word_targets=batch["word_targets"],
            word_mask=batch["word_mask"]
        )

        # ---------- frozen LM loss (GEN tasks only) ----------
        loss_lm = None
        if self.lm is not None:
            # compute only if this is a gen row
            if float(batch["word_mask"].item()) > 0.0:
                loss_lm = self._frozen_lm_loss(
                    word_logits=dec["word_logits"],
                    q_text=batch.get("q_text", ""),
                    t_text=batch.get("t_text", ""),
                    device=batch["e_seq"].device
                )

        loss_total = (
            enc["loss_action"]
            + enc["loss_node"]
            + enc["loss_task"]
            + loss_user
            + loss_item
            + dec["loss_word"]
        )

        if loss_lm is not None:
            loss_total = loss_total + self.lm_loss_weight * loss_lm

        return {
            "loss_total": loss_total,

            "loss_action": enc["loss_action"],
            "loss_node": enc["loss_node"],
            "loss_task": enc["loss_task"],
            "loss_user": loss_user,
            "loss_item": loss_item,
            "loss_word": dec["loss_word"],
            "loss_lm": (loss_lm if loss_lm is not None else torch.tensor(0.0, device=h_final.device)),

            "act_logits": enc["act_logits"],
            "node_logits": enc["node_logits"],

            "task_logits": enc["task_logits"],
            "P_task": enc["P_task"],

            "user_logits": user_logits,
            "item_logits": item_logits,
            "word_logits": dec["word_logits"],

            "h_final": enc["h_final"]
        }


In [ ]:
# =========================================================
# REPLACEMENT: CELL 17/17.5/17.6 — Build Frozen LM first, then model, then attach once
# =========================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- Frozen decoder-only LM ----
LM_NAME = "distilgpt2"  # ~82M params
lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
if lm_tokenizer.pad_token is None:
    lm_tokenizer.pad_token = lm_tokenizer.eos_token

lm_model = AutoModelForCausalLM.from_pretrained(LM_NAME)
lm_model.eval()
for p in lm_model.parameters():
    p.requires_grad = False
lm_model = lm_model.to(device)

print("Frozen LM ready:", LM_NAME, "| d_model:", lm_model.get_input_embeddings().embedding_dim)

# ---- Instantiate Multi-task model (NO LM attachment yet) ----
vocab_sizes = {
    "actions": len(actions2idx),
    "nodes": len(node2idx),
}

encoder = UniHisSSMEncoder(
    vocab_sizes=vocab_sizes,
    d_model=D_MODEL,
    d_state=64,
    n_tasks=n_tasks,
    d_qtext=D_MODEL,
    known_task=False
)

model = UniHisMultiTaskModel(
    encoder=encoder,
    d_qtext=D_MODEL,
    n_tasks=n_tasks,
    n_users_small=len(user2idx_small),
    n_items=len(item2idx),
    n_words=len(word2idx),
    rwkv_layers=2
).to(device)

# ---- Attach frozen LM + build differentiable bridge (lm_* params) ----
model.set_frozen_lm(
    lm_model=lm_model,
    lm_tokenizer=lm_tokenizer,
    hint_len=8,
    d_bridge=128,
    word_temp=1.0,
    loss_weight=1.0,
    max_prompt_tokens=256,
    max_target_tokens=256,
)

# IMPORTANT: model is already on device; LM is already on device
print("Model ready (LM attached).")

def count_parameters(m):
    total = 0
    trainable = 0
    frozen = 0
    for _, p in m.named_parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
        else:
            frozen += n
    return total, trainable, frozen

total_params, trainable_params, frozen_params = count_parameters(model)
print("\n================ MODEL PARAMETER SUMMARY ================")
print(f"Total parameters          : {total_params:,}")
print(f"Trainable parameters      : {trainable_params:,}")
print(f"Frozen parameters         : {frozen_params:,}")


In [ ]:
# # =========================================================
# # CELL 17 — Instantiate Model + Optimizers (base + lm-bridge)
# # (DROP-IN REPLACEMENT)
# # =========================================================

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Device:", device)

# vocab_sizes = {
#     "actions": len(actions2idx),
#     "nodes": len(node2idx),
# }

# encoder = UniHisSSMEncoder(
#     vocab_sizes=vocab_sizes,
#     d_model=D_MODEL,
#     d_state=64,
#     n_tasks=n_tasks,
#     d_qtext=D_MODEL,
#     known_task=False
# )

# model = UniHisMultiTaskModel(
#     encoder=encoder,
#     d_qtext=D_MODEL,
#     n_tasks=n_tasks,
#     n_users_small=len(user2idx_small),
#     n_items=len(item2idx),
#     n_words=len(word2idx),
#     rwkv_layers=2
# ).to(device)

# # Attach frozen LM + build bridge params (lm_*)
# model.set_frozen_lm(
#     lm_model=lm_model,
#     lm_tokenizer=lm_tokenizer,
#     hint_len=8,
#     d_bridge=128,
#     word_temp=1.0,
#     loss_weight=1.0,
#     max_prompt_tokens=256,
#     max_target_tokens=256,
# )

# # Two optimizers: base (resumable), lm-bridge (new)
# optim_base = torch.optim.AdamW(model.base_param_list(), lr=3e-4, weight_decay=1e-2)
# optim_lm   = torch.optim.AdamW(model.lm_param_list(),   lr=3e-4, weight_decay=1e-2)

# print("Model ready.")
# print("Base params:", sum(p.numel() for p in model.base_param_list()))
# print("LM-bridge params:", sum(p.numel() for p in model.lm_param_list()))

# def count_parameters(m):
#     total = 0
#     trainable = 0
#     frozen = 0
#     for _, p in m.named_parameters():
#         n = p.numel()
#         total += n
#         if p.requires_grad:
#             trainable += n
#         else:
#             frozen += n
#     return total, trainable, frozen

# total_params, trainable_params, frozen_params = count_parameters(model)
# print("\n================ MODEL PARAMETER SUMMARY ================")
# print(f"Total deployable parameters : {total_params:,}")
# print(f"Trainable parameters       : {trainable_params:,}")
# print(f"Frozen parameters          : {frozen_params:,}")


In [107]:
# # =========================================================
# # CELL 17.5 — Frozen Decoder-Only LM (50M class)
# # =========================================================

# from transformers import AutoTokenizer, AutoModelForCausalLM

# LM_NAME = "distilgpt2"   # ~82M params, smallest stable GPT-2

# lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
# if lm_tokenizer.pad_token is None:
#     lm_tokenizer.pad_token = lm_tokenizer.eos_token

# lm_model = AutoModelForCausalLM.from_pretrained(LM_NAME)
# lm_model.eval()
# for p in lm_model.parameters():
#     p.requires_grad = False


In [108]:
# # =========================================================
# # CELL 17.6 — Attach frozen LM + build differentiable bridge
# # =========================================================

# model.set_frozen_lm(
#     lm_model=lm_model,
#     lm_tokenizer=lm_tokenizer,
#     hint_len=8,
#     d_bridge=128,
#     word_temp=1.0,
#     loss_weight=1.0,
#     max_prompt_tokens=256,
#     max_target_tokens=256
# )

# # IMPORTANT: move model AFTER attaching LM
# model = model.to(device)
# lm_model = lm_model.to(device)


In [ ]:
# =========================================================
# NEW CELL — Freeze policy for Epoch 5 finetuning
#   Freeze: encoder + item head
#   Train : user head + RWKV word decoder + LM bridge (lm_*)
# =========================================================

def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

# Freeze entire encoder (includes action/node/task heads inside encoder)
set_requires_grad(model.encoder, False)

# Freeze item head only
set_requires_grad(model.heads.item_head, False)

# Train user head
set_requires_grad(model.heads.user_head, True)

# Train RWKV word decoder
set_requires_grad(model.word_decoder, True)

# Train LM-bridge params only (NOT the frozen LM itself)
# lm_word_soft_emb (Parameter) and lm_hint_proj (Linear) are trainable by default.
# Ensure they are trainable explicitly:
if model.lm_word_soft_emb is not None:
    model.lm_word_soft_emb.requires_grad = True
set_requires_grad(model.lm_hint_proj, True)

# Safety: confirm frozen LM is still frozen
for p in model.lm.parameters():
    assert p.requires_grad is False

# Report what will train
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
frozen = [(n, p.numel()) for n, p in model.named_parameters() if not p.requires_grad]

print("Trainable parameter groups (name, #params):")
for n, k in trainable[:50]:
    print("  ", n, k)
if len(trainable) > 50:
    print("  ... total trainable tensors:", len(trainable))

print("\nTotal trainable params:", sum(k for _, k in trainable))
print("Total frozen params   :", sum(k for _, k in frozen))


In [21]:
# =========================================================
# CELL 18 — Utilities: Top-K decoding with probabilities (User/Item) + Word decoding
# Purpose:
#   - Produce top-15 predictions with probs
# Inputs:
#   - logits tensors
# Outputs:
#   - list[(token, prob)]
# =========================================================

@torch.no_grad()
def topk_probs(logits, idx2tok, k=15):
    probs = torch.softmax(logits, dim=-1)  # (1,V)
    topv, topi = torch.topk(probs, k=min(k, probs.size(-1)), dim=-1)
    topv = topv[0].tolist()
    topi = topi[0].tolist()
    return [(idx2tok[i], float(p)) for i, p in zip(topi, topv)]

@torch.no_grad()
def topk_words_from_logits(word_logits, idx2word, k=20, threshold=0.2):
    # multi-label: use sigmoid
    scores = torch.sigmoid(word_logits)[0]  # (V,)
    keep = scores >= threshold
    idxs = torch.nonzero(keep).squeeze(-1)
    if idxs.numel() == 0:
        # fallback to top-k anyway
        topv, topi = torch.topk(scores, k=min(k, scores.numel()))
        return [(idx2word[int(i)], float(v)) for i, v in zip(topi.tolist(), topv.tolist())]
    vals = scores[idxs]
    kk = min(k, vals.numel())
    topv, order = torch.topk(vals, kk)
    picked = idxs[order]
    return [(idx2word[int(i)], float(v)) for i, v in zip(picked.tolist(), topv.tolist())]


In [ ]:
# =========================================================
# REPLACEMENT: CELL 20 — Training Loop starting Epoch 5
#   - Loads epoch_4.pt
#   - Freezes encoder + item head
#   - Trains only user head + word decoder + LM bridge
# =========================================================

from tqdm.auto import tqdm
import os, re, torch
from collections import defaultdict

CKPT_DIR = "/teamspace/studios/this_studio/checkpoints_v2"
os.makedirs(CKPT_DIR, exist_ok=True)

EPOCHS = 7
CLIP_NORM = 0.75
SAVE_EVERY_ROWS = 12000
SAVE_EVERY_EPOCH = True

BASE_CKPT_PATH = os.path.join(CKPT_DIR, "epoch_4.pt")
if not os.path.exists(BASE_CKPT_PATH):
    raise FileNotFoundError(f"Base checkpoint not found: {BASE_CKPT_PATH}")

LOSS_KEYS_ORDER = [
    "loss_total",
    "loss_action",
    "loss_node",
    "loss_task",
    "loss_user",
    "loss_item",
    "loss_word",
    "loss_lm",
]

LOSS_DECIMALS = {
    "loss_total": 4,
    "loss_user": 4,
    "loss_item": 4,
    "loss_word": 4,
    "loss_lm": 4,
    "loss_action": 2,
    "loss_node": 2,
    "loss_task": 2,
}

def running_mean(x, n):
    return x / max(1, n)

def fmt_loss(name, value, n_ok):
    if n_ok <= 0:
        return "na"
    dec = LOSS_DECIMALS.get(name, 2)
    return f"{running_mean(value, n_ok):.{dec}f}"

def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

# ---- load weights from epoch_4 ----
ckpt = torch.load(BASE_CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model"], strict=False)
model.to(device)

# ---- apply freeze policy again (robust) ----
set_requires_grad(model.encoder, False)
set_requires_grad(model.heads.item_head, False)
set_requires_grad(model.heads.user_head, True)
set_requires_grad(model.word_decoder, True)
if model.lm_word_soft_emb is not None:
    model.lm_word_soft_emb.requires_grad = True
set_requires_grad(model.lm_hint_proj, True)

# ---- optimizer only over trainable params ----
trainable_params = [p for p in model.parameters() if p.requires_grad]
if len(trainable_params) == 0:
    raise RuntimeError("No trainable parameters found. Freeze policy likely wrong.")

optim = torch.optim.AdamW(trainable_params, lr=3e-4, weight_decay=1e-2)

# ---- epoch start: FORCE start from 5 ----
start_epoch = 5
start_row_i = 0
global_step = int(ckpt.get("global_step", 0))

print("STARTING FINETUNE")
print("Loaded ckpt:", BASE_CKPT_PATH)
print("Start epoch:", start_epoch, "| Start row:", start_row_i, "| global_step:", global_step)
print("Trainable tensors:", sum(1 for p in model.parameters() if p.requires_grad))
print("Trainable params :", sum(p.numel() for p in model.parameters() if p.requires_grad))

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()

    sums = {k: 0.0 for k in LOSS_KEYS_ORDER}
    steps_ok = 0
    steps_skip = 0

    row_start = start_row_i if epoch == start_epoch else 0
    pbar = tqdm(range(row_start, len(train_df)), desc=f"Epoch {epoch} (row {row_start}→)")

    for row_i in pbar:
        row = train_df.iloc[row_i]
        batch = build_row_tensors(row, device)

        # inject text for LM (safe even if unused)
        if isinstance(batch, dict) and batch.get("ok", False):
            batch["q_text"] = id2text.get(batch.get("qid", ""), "")
            batch["t_text"] = id2text.get(batch.get("tid", ""), "")

        if not batch["ok"]:
            steps_skip += 1
            continue

        out = model(batch)
        loss = out["loss_total"]

        if not torch.isfinite(loss):
            optim.zero_grad(set_to_none=True)
            steps_skip += 1
            continue

        optim.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(trainable_params, CLIP_NORM)
        optim.step()

        steps_ok += 1
        global_step += 1

        for k in LOSS_KEYS_ORDER:
            if k in out:
                sums[k] += float(out[k].detach().item())

        if (row_i + 1) % SAVE_EVERY_ROWS == 0:
            step_path = os.path.join(CKPT_DIR, f"step_{epoch}_{row_i+1}.pt")
            torch.save({
                "epoch": epoch,
                "row_i_next": row_i + 1,
                "global_step": global_step,
                "model": model.state_dict(),
            }, step_path)

        pbar.set_postfix({
            "L":   fmt_loss("loss_total", sums["loss_total"], steps_ok),
            "La":  fmt_loss("loss_action", sums["loss_action"], steps_ok),
            "Ln":  fmt_loss("loss_node", sums["loss_node"], steps_ok),
            "Lt":  fmt_loss("loss_task", sums["loss_task"], steps_ok),
            "Lu":  fmt_loss("loss_user", sums["loss_user"], steps_ok),
            "Li":  fmt_loss("loss_item", sums["loss_item"], steps_ok),
            "Lw":  fmt_loss("loss_word", sums["loss_word"], steps_ok),
            "Llm": fmt_loss("loss_lm", sums["loss_lm"], steps_ok),
            "skip": steps_skip,
            "row": row_i
        })

    if SAVE_EVERY_EPOCH:
        epoch_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pt")
        torch.save({
            "epoch": epoch + 1,
            "row_i_next": 0,
            "global_step": global_step,
            "model": model.state_dict(),
            "epoch_losses": {k: running_mean(v, steps_ok) for k, v in sums.items()},
            "steps_ok": steps_ok,
            "steps_skip": steps_skip,
        }, epoch_path)

    print("\nEpoch", epoch, "SUMMARY")
    for k in LOSS_KEYS_ORDER:
        if steps_ok > 0:
            dec = LOSS_DECIMALS.get(k, 2)
            print(f"{k:12s}: {running_mean(sums[k], steps_ok):.{dec}f}")
        else:
            print(f"{k:12s}: na")
    print("steps_ok:", steps_ok, "steps_skip:", steps_skip)

    start_row_i = 0


In [ ]:
import torch, math, numpy as np
from collections import defaultdict
from tqdm import tqdm

# ---------------------------------------------------------
# Load trained model (CPU safe)
# ---------------------------------------------------------

CKPT_PATH = "/teamspace/studios/this_studio/checkpoints_v2/epoch_7.pt"

ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=False)
model.to(device)
model.eval()

print("Loaded:", CKPT_PATH, "epoch:", ckpt.get("epoch"), "global_step:", ckpt.get("global_step"))


# =========================================================
# Ranking Metrics
# =========================================================

def mrr_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / math.log2(i + 1)
    return 0.0


# =========================================================
# ROUGE (token based, same as your training side)
# =========================================================

def rouge1_f1(pred_tokens, ref_tokens):
    if len(pred_tokens)==0 and len(ref_tokens)==0: return 1.0
    if len(pred_tokens)==0 or len(ref_tokens)==0: return 0.0
    from collections import Counter
    pc = Counter(pred_tokens)
    rc = Counter(ref_tokens)
    overlap = sum((pc & rc).values())
    p = overlap / max(1,len(pred_tokens))
    r = overlap / max(1,len(ref_tokens))
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def lcs_length(a,b):
    n,m=len(a),len(b)
    dp=[0]*(m+1)
    for i in range(1,n+1):
        prev=0
        for j in range(1,m+1):
            tmp=dp[j]
            if a[i-1]==b[j-1]:
                dp[j]=prev+1
            else:
                dp[j]=max(dp[j],dp[j-1])
            prev=tmp
    return dp[m]

def rougeL_f1(pred_tokens, ref_tokens):
    if len(pred_tokens)==0 and len(ref_tokens)==0: return 1.0
    if len(pred_tokens)==0 or len(ref_tokens)==0: return 0.0
    lcs=lcs_length(pred_tokens,ref_tokens)
    p=lcs/max(1,len(pred_tokens))
    r=lcs/max(1,len(ref_tokens))
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def tokenize_simple(x):
    return tokenize_words(x or "")


# =========================================================
# LM generation WITH RWKV hint embeddings
# =========================================================

@torch.no_grad()
def lm_generate_with_hints(word_logits, q_text,
                           max_new_tokens=80,
                           temperature=0.8,
                           top_p=0.95):

    q_text=(q_text or "").strip()
    if len(q_text)==0:
        return ""

    prompt="Context:\n"+q_text+"\n\nInstruction: Write the best possible response.\nResponse:\n"

    tok=model.lm_tokenizer(prompt,return_tensors="pt",
                           truncation=True,
                           max_length=model.lm_max_prompt_tokens,
                           add_special_tokens=False)
    prompt_ids=tok["input_ids"].to(device)
    E=model.lm.get_input_embeddings()
    prompt_embeds=E(prompt_ids)

    p_words=torch.softmax(word_logits/model.lm_word_temp,dim=-1)
    e_bridge=p_words@model.lm_word_soft_emb
    hint_flat=model.lm_hint_proj(e_bridge)
    hint_embeds=hint_flat.view(1,model.lm_hint_len,model.lm_d_model)

    inputs_embeds=torch.cat([prompt_embeds,hint_embeds],dim=1)
    attn=torch.ones((1,inputs_embeds.size(1)),device=device)

    gen_ids=model.lm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=model.lm_tokenizer.pad_token_id,
        eos_token_id=model.lm_tokenizer.eos_token_id
    )

    out=model.lm_tokenizer.decode(gen_ids[0],skip_special_tokens=True)
    return out.split("Response:",1)[-1].strip() if "Response:" in out else out.strip()


# =========================================================
# Full Evaluation
# =========================================================

TOPK=15
records=[]
metrics=defaultdict(list)

model.eval()

with torch.no_grad():
    for i in tqdm(range(len(test_df[:150])),desc="Inference"):
        row=test_df.iloc[i]
        batch=build_row_tensors(row,device)
        if not batch["ok"]:
            continue

        batch["q_text"]=id2text.get(batch["qid"],"")
        batch["t_text"]=id2text.get(batch["tid"],"")

        out=model(batch)
        task=batch["task"]

        rec={"task":task,"qid":batch["qid"],"tid":batch["tid"],"userid":batch["userid_raw"]}

        # ----- USER -----
        if task==USER_TASK:
            probs=torch.softmax(out["user_logits"],dim=-1)[0]
            topv,topi=torch.topk(probs,k=min(TOPK,probs.numel()))
            gold=int(batch["user_labels"].item())
            rec["gold_user"]=idx2user_small[gold]
            rec["top15_user"]=[(idx2user_small[int(i)],float(v)) for i,v in zip(topi,topv)]
            metrics["user_mrr15"].append(mrr_at_k(topi.tolist(),gold))
            metrics["user_ndcg15"].append(ndcg_at_k(topi.tolist(),gold))

        # ----- ITEM -----
        if task==ITEM_TASK:
            probs=torch.softmax(out["item_logits"],dim=-1)[0]
            topv,topi=torch.topk(probs,k=min(TOPK,probs.numel()))
            gold=int(batch["item_labels"].item())
            rec["gold_item"]=idx2item[gold]
            rec["top15_item"]=[(idx2item[int(i)],float(v)) for i,v in zip(topi,topv)]
            metrics["item_mrr15"].append(mrr_at_k(topi.tolist(),gold))
            metrics["item_ndcg15"].append(ndcg_at_k(topi.tolist(),gold))

        # ----- GENERATION (LM) -----
        if task in GEN_TASKS_WORD:
            lm_text=lm_generate_with_hints(out["word_logits"],batch["q_text"])
            rec["lm_pred_text"]=lm_text
            rec["gold_text"]=batch["t_text"]

            p=tokenize_simple(lm_text)
            r=tokenize_simple(batch["t_text"])
            metrics[f"{task}_rouge1"].append(rouge1_f1(p,r))
            metrics[f"{task}_rougeL"].append(rougeL_f1(p,r))

        records.append(rec)

results_df=pd.DataFrame(records)

print("\n=== USER ===")
print("MRR@15 :",np.mean(metrics["user_mrr15"]))
print("NDCG@15:",np.mean(metrics["user_ndcg15"]))

print("\n=== ITEM ===")
print("MRR@15 :",np.mean(metrics["item_mrr15"]))
print("NDCG@15:",np.mean(metrics["item_ndcg15"]))

for t in GEN_TASKS_WORD:
    print("\n===",t,"===")
    print("ROUGE-1:",np.mean(metrics[f"{t}_rouge1"]))
    print("ROUGE-L:",np.mean(metrics[f"{t}_rougeL"]))

results_df.head()


In [ ]:
# =========================================================
# Inspect 5 Dialogue + 5 Summary generations
# =========================================================

def show_gen_samples(task_name, n=5):
    sub = results_df[results_df["task"] == task_name].head(n)

    print("\n" + "="*120)
    print("TASK:", task_name, "| Showing", len(sub), "examples")
    print("="*120)

    for i, r in sub.iterrows():
        print("\n" + "-"*120)
        print("QID:", r["qid"], " | TID:", r["tid"])
        print("\n[QUERY TEXT]")
        print(r.get("query_text", "")[:1500])

        print("\n[GOLD TEXT]")
        print(r.get("gold_text", "")[:1500])

        print("\n[LM GENERATED TEXT]")
        print(r.get("lm_pred_text", "")[:1500])

        # Compute ROUGE live for readability
        pred = tokenize_simple(r.get("lm_pred_text",""))
        gold = tokenize_simple(r.get("gold_text",""))
        print("\n[ROUGE]")
        print("  ROUGE-1:", rouge1_f1(pred, gold))
        print("  ROUGE-L:", rougeL_f1(pred, gold))


show_gen_samples("dialogue_generation", n=5)
show_gen_samples("personalized_summarization", n=5)


In [ ]:
import torch

@torch.no_grad()
def lm_teacher_forced_debug(batch, max_steps=30):
    """
    Shows how well the LM predicts gold tokens when conditioned on:
      prompt + RWKV hint embeddings + gold prefix
    """

    q_text = batch["q_text"]
    t_text = batch["t_text"]

    if len(q_text.strip()) == 0 or len(t_text.strip()) == 0:
        print("Empty query or target.")
        return

    # ---- build prompt ----
    prompt = (
        "Context:\n"
        + q_text
        + "\n\nInstruction: Write the best possible response.\nResponse:\n"
    )

    tok_prompt = model.lm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=model.lm_max_prompt_tokens,
        add_special_tokens=False
    )

    tok_target = model.lm_tokenizer(
        t_text,
        return_tensors="pt",
        truncation=True,
        max_length=model.lm_max_target_tokens,
        add_special_tokens=False
    )

    prompt_ids = tok_prompt["input_ids"].to(device)
    target_ids = tok_target["input_ids"].to(device)

    E = model.lm.get_input_embeddings()

    # ---- prompt embeddings ----
    prompt_embeds = E(prompt_ids)

    # ---- RWKV hint embeddings ----
    p_words = torch.softmax(batch["word_logits"] / model.lm_word_temp, dim=-1)
    e_bridge = p_words @ model.lm_word_soft_emb
    hint_flat = model.lm_hint_proj(e_bridge)
    hint_embeds = hint_flat.view(1, model.lm_hint_len, model.lm_d_model)

    print("\n===== TEACHER FORCED LM DEBUG =====")
    print("PROMPT:", prompt[:200])
    print("TARGET TEXT:", t_text[:200])
    print("="*80)

    cur_embeds = torch.cat([prompt_embeds, hint_embeds], dim=1)

    max_steps = min(max_steps, target_ids.size(1))

    for t in range(max_steps):
        # Run LM
        out = model.lm(inputs_embeds=cur_embeds)
        logits = out.logits[:, -1, :]   # last token distribution

        gold_token = target_ids[0, t].item()
        gold_str = model.lm_tokenizer.decode([gold_token])

        probs = torch.softmax(logits, dim=-1)[0]
        gold_prob = probs[gold_token].item()

        topv, topi = torch.topk(probs, 5)
        top_preds = [(model.lm_tokenizer.decode([i.item()]), float(v)) for i,v in zip(topi,topv)]

        print(f"\nStep {t+1}")
        print("Gold token:", repr(gold_str), "  P =", gold_prob)
        print("Top-5 predictions:")
        for tok,p in top_preds:
            print("   ", repr(tok), f"{p:.4f}")

        # append gold token (teacher forcing)
        next_embed = E(target_ids[:, t:t+1])
        cur_embeds = torch.cat([cur_embeds, next_embed], dim=1)

    print("\n===================================")
row = test_df[test_df["task"]=="dialogue_generation"].iloc[0]
batch = build_row_tensors(row, device)
batch["q_text"] = id2text.get(batch["qid"], "")
batch["t_text"] = id2text.get(batch["tid"], "")

out = model(batch)
batch["word_logits"] = out["word_logits"]

lm_teacher_forced_debug(batch, max_steps=20)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LM_NAME = "distilgpt2"

lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
if lm_tokenizer.pad_token is None:
    lm_tokenizer.pad_token = lm_tokenizer.eos_token

lm_model = AutoModelForCausalLM.from_pretrained(LM_NAME)
lm_model.eval()
for p in lm_model.parameters():
    p.requires_grad = False

lm_model = lm_model.to(device)
print("Frozen LM ready on:", next(lm_model.parameters()).device)


In [ ]:
# IMPORTANT: this recreates lm_word_soft_emb and lm_hint_proj with NEW shapes
# hint_len: 64 (new), d_bridge unchanged (128), etc.

model.set_frozen_lm(
    lm_model=lm_model,
    lm_tokenizer=lm_tokenizer,
    hint_len=64,          # <-- NEW (was 8)
    d_bridge=128,
    word_temp=1.0,
    loss_weight=1.0,
    max_prompt_tokens=256,
    max_target_tokens=256
)

model = model.to(device)
model.eval()

print("Attached LM with hint_len =", model.lm_hint_len)
print("lm_word_soft_emb:", tuple(model.lm_word_soft_emb.shape))
print("lm_hint_proj.weight:", tuple(model.lm_hint_proj.weight.shape))
print("lm_hint_proj.bias:", tuple(model.lm_hint_proj.bias.shape))


In [ ]:
import torch
import os

CKPT_E7 = "/teamspace/studios/this_studio/checkpoints_v2/epoch_7.pt"
assert os.path.exists(CKPT_E7), f"Missing checkpoint: {CKPT_E7}"

ckpt = torch.load(CKPT_E7, map_location="cpu")
sd = ckpt["model"]

# -------------------------------
# 1) Extract old LM bridge weights
# -------------------------------
old_lm_word_soft_emb = sd.get("lm_word_soft_emb", None)
old_lm_hint_w = sd.get("lm_hint_proj.weight", None)
old_lm_hint_b = sd.get("lm_hint_proj.bias", None)

# -------------------------------
# 2) Remove LM-bridge tensors from checkpoint
#    (so load_state_dict won't choke on size mismatch)
# -------------------------------
bad_keys = [k for k in sd.keys() if k.startswith("lm_")]
for k in bad_keys:
    sd.pop(k)

print("Removed LM bridge keys from checkpoint:", bad_keys)

# -------------------------------
# 3) Load everything else
# -------------------------------
missing, unexpected = model.load_state_dict(sd, strict=False)

print("Loaded epoch_7 core weights")
print("Missing keys (count):", len(missing))
print("Unexpected keys (count):", len(unexpected))

# -------------------------------
# 4) Restore lm_word_soft_emb (same shape, safe)
# -------------------------------
if old_lm_word_soft_emb is not None:
    model.lm_word_soft_emb.data.copy_(old_lm_word_soft_emb.to(model.lm_word_soft_emb.device))
    print("Restored lm_word_soft_emb")

# -------------------------------
# 5) Transplant old 8-hint lm_hint_proj into first slots of new 64-hint
# -------------------------------
if old_lm_hint_w is not None and old_lm_hint_b is not None:
    old_lm_hint_w = old_lm_hint_w.cpu()
    old_lm_hint_b = old_lm_hint_b.cpu()

    new_w = model.lm_hint_proj.weight.data
    new_b = model.lm_hint_proj.bias.data

    n_old = old_lm_hint_w.shape[0]   # = 8 * d_lm

    assert new_w.shape[1] == old_lm_hint_w.shape[1], "d_bridge mismatch"
    assert new_b.shape[0] >= n_old, "new hint proj too small"

    new_w[:n_old].copy_(old_lm_hint_w.to(new_w.device))
    new_b[:n_old].copy_(old_lm_hint_b.to(new_b.device))

    print(f"Transplanted old lm_hint_proj: copied {n_old} rows into new 64-hint bridge")

# -------------------------------
# 6) Move model to device and train mode
# -------------------------------
model = model.to(device)
model.train()

print("Epoch-7 checkpoint successfully adapted to hint_len=64")


In [ ]:
def set_requires_grad(m, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag

# Freeze EVERYTHING first
set_requires_grad(model, False)

# Unfreeze ONLY:
# (1) user head
set_requires_grad(model.heads.user_head, True)

# (2) RWKV word decoder
set_requires_grad(model.word_decoder, True)

# (3) LM bridge params
model.lm_word_soft_emb.requires_grad = True
set_requires_grad(model.lm_hint_proj, True)

# Explicitly keep encoder frozen and item head frozen (already frozen by global freeze)
set_requires_grad(model.encoder, False)
set_requires_grad(model.heads.item_head, False)

# Sanity print: trainable parameter names
trainable = [(n, p.numel()) for n,p in model.named_parameters() if p.requires_grad]
print("Trainable param groups:", len(trainable))
print("Trainable total params:", sum(x for _,x in trainable))
print("\nSample trainable names:")
for n,_ in trainable[:20]:
    print("  ", n)


In [ ]:
FT_TASKS = set(GEN_TASKS_WORD) | {USER_TASK}
finetune_df = train_df[train_df["task"].astype(str).str.strip().isin(FT_TASKS)].reset_index(drop=True)

print("Finetune rows:", len(finetune_df))
print("Task counts:\n", finetune_df["task"].value_counts())


In [ ]:
import torch
from torch.optim import AdamW

# Collect trainable params only
params = []
params += list(model.heads.user_head.parameters())
params += list(model.word_decoder.parameters())
params += [model.lm_word_soft_emb]
params += list(model.lm_hint_proj.parameters())

# Remove any accidental duplicates
seen = set()
uniq = []
for p in params:
    if id(p) not in seen:
        uniq.append(p)
        seen.add(id(p))

optim_ft = AdamW(uniq, lr=1e-4, weight_decay=1e-2)

print("Finetune optimizer params:", sum(p.numel() for p in uniq))


In [ ]:
import os
from tqdm.auto import tqdm

FT_CKPT_DIR = "/teamspace/studios/this_studio/checkpoints_ft64"
os.makedirs(FT_CKPT_DIR, exist_ok=True)

FT_EPOCHS = 2          # start small; increase after verifying improvements
CLIP_NORM = 0.75
SAVE_EVERY_ROWS = 12000

# Loss weights (strict defaults)
LM_W   = 10.0   # <-- make LM supervision dominant
USER_W = 1.0
WORD_W = 0.2    # small stabilizer

global_step_ft = 0

model.train()

for epoch in range(1, FT_EPOCHS + 1):
    sums = defaultdict(float)
    steps_ok = 0
    steps_skip = 0

    pbar = tqdm(range(len(finetune_df)), desc=f"FT64 Epoch {epoch}")

    for row_i in pbar:
        row = finetune_df.iloc[row_i]
        batch = build_row_tensors(row, device)

        if not batch.get("ok", False):
            steps_skip += 1
            continue

        # Inject texts for LM teacher forcing (required)
        batch["q_text"] = id2text.get(batch["qid"], "")
        batch["t_text"] = id2text.get(batch["tid"], "")

        out = model(batch)

        # Teacher-forced LM loss exists only when word_mask=1 inside model.forward
        loss_lm = out["loss_lm"]
        loss_user = out["loss_user"]
        loss_word = out["loss_word"]

        # STRICT finetune objective
        loss = LM_W * loss_lm + USER_W * loss_user + WORD_W * loss_word

        if not torch.isfinite(loss):
            optim_ft.zero_grad(set_to_none=True)
            steps_skip += 1
            continue

        optim_ft.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            CLIP_NORM
        )

        optim_ft.step()

        steps_ok += 1
        global_step_ft += 1

        # Track
        sums["loss"] += float(loss.item())
        sums["loss_lm"] += float(loss_lm.item())
        sums["loss_user"] += float(loss_user.item())
        sums["loss_word"] += float(loss_word.item())

        if (row_i + 1) % SAVE_EVERY_ROWS == 0:
            path = os.path.join(FT_CKPT_DIR, f"step_ft64_e{epoch}_{row_i+1}.pt")
            torch.save({
                "epoch": epoch,
                "row_i_next": row_i + 1,
                "global_step_ft": global_step_ft,
                "model": model.state_dict(),
                "LM_W": LM_W, "USER_W": USER_W, "WORD_W": WORD_W
            }, path)

        pbar.set_postfix({
            "L":   sums["loss"] / max(1, steps_ok),
            "Llm": sums["loss_lm"] / max(1, steps_ok),
            "Lu":  sums["loss_user"] / max(1, steps_ok),
            "Lw":  sums["loss_word"] / max(1, steps_ok),
            "skip": steps_skip
        })

    # Epoch save
    ep_path = os.path.join(FT_CKPT_DIR, f"epoch_ft64_{epoch}.pt")
    torch.save({
        "epoch": epoch + 1,
        "row_i_next": 0,
        "global_step_ft": global_step_ft,
        "model": model.state_dict(),
        "epoch_means": {k: v / max(1, steps_ok) for k,v in sums.items()},
        "steps_ok": steps_ok,
        "steps_skip": steps_skip,
        "LM_W": LM_W, "USER_W": USER_W, "WORD_W": WORD_W
    }, ep_path)

    print("\nFT64 Epoch", epoch, "summary:",
          {k: round(v / max(1, steps_ok), 6) for k,v in sums.items()},
          "ok:", steps_ok, "skip:", steps_skip)

model.eval()


In [ ]:
FT_CKPT = "/teamspace/studios/this_studio/checkpoints_ft64/epoch_ft64_2.pt"
ckpt2 = torch.load(FT_CKPT, map_location="cpu")
model.load_state_dict(ckpt2["model"], strict=False)
model.to(device)
model.eval()
print("Loaded finetuned:", FT_CKPT)


In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import math
import torch

TOPK = 15

def mean_or_nan(x):
    return float(np.mean(x)) if len(x) else float("nan")

def mrr_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / math.log2(i + 1)
    return 0.0

def rouge1_f1(pred_tokens, ref_tokens):
    if len(pred_tokens)==0 and len(ref_tokens)==0: return 1.0
    if len(pred_tokens)==0 or len(ref_tokens)==0: return 0.0
    from collections import Counter
    pc = Counter(pred_tokens)
    rc = Counter(ref_tokens)
    overlap = sum((pc & rc).values())
    p = overlap / max(1,len(pred_tokens))
    r = overlap / max(1,len(ref_tokens))
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def lcs_length(a,b):
    n,m=len(a),len(b)
    dp=[0]*(m+1)
    for i in range(1,n+1):
        prev=0
        for j in range(1,m+1):
            tmp=dp[j]
            if a[i-1]==b[j-1]:
                dp[j]=prev+1
            else:
                dp[j]=max(dp[j],dp[j-1])
            prev=tmp
    return dp[m]

def rougeL_f1(pred_tokens, ref_tokens):
    if len(pred_tokens)==0 and len(ref_tokens)==0: return 1.0
    if len(pred_tokens)==0 or len(ref_tokens)==0: return 0.0
    lcs=lcs_length(pred_tokens,ref_tokens)
    p=lcs/max(1,len(pred_tokens))
    r=lcs/max(1,len(ref_tokens))
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def tokenize_simple(x):
    return tokenize_words(x or "")

@torch.no_grad()
def lm_generate_with_hints(word_logits, q_text, max_new_tokens=80, temperature=0.8, top_p=0.95):
    q_text=(q_text or "").strip()
    if len(q_text)==0:
        return ""

    prompt="Context:\n"+q_text+"\n\nInstruction: Write the best possible response.\nResponse:\n"

    tok=model.lm_tokenizer(prompt,return_tensors="pt",
                           truncation=True,
                           max_length=model.lm_max_prompt_tokens,
                           add_special_tokens=False)
    prompt_ids=tok["input_ids"].to(device)
    E=model.lm.get_input_embeddings()
    prompt_embeds=E(prompt_ids)

    p_words=torch.softmax(word_logits/model.lm_word_temp,dim=-1)
    e_bridge=p_words@model.lm_word_soft_emb
    hint_flat=model.lm_hint_proj(e_bridge)
    hint_embeds=hint_flat.view(1,model.lm_hint_len,model.lm_d_model)

    inputs_embeds=torch.cat([prompt_embeds,hint_embeds],dim=1)
    attn=torch.ones((1,inputs_embeds.size(1)),device=device)

    gen_ids=model.lm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=model.lm_tokenizer.pad_token_id,
        eos_token_id=model.lm_tokenizer.eos_token_id
    )

    out=model.lm_tokenizer.decode(gen_ids[0],skip_special_tokens=True)
    return out.split("Response:",1)[-1].strip() if "Response:" in out else out.strip()

records=[]
metrics=defaultdict(list)

model.eval()
with torch.no_grad():
    for i in tqdm(range(len(test_df[:150])), desc="Eval"):
        row=test_df.iloc[i]
        batch=build_row_tensors(row,device)
        if not batch.get("ok",False):
            continue

        batch["q_text"]=id2text.get(batch["qid"],"")
        batch["t_text"]=id2text.get(batch["tid"],"")

        out=model(batch)
        task=batch["task"]

        rec={"task":task,"qid":batch["qid"],"tid":batch["tid"],"userid":batch["userid_raw"]}

        if task==USER_TASK:
            probs=torch.softmax(out["user_logits"],dim=-1)[0]
            topv,topi=torch.topk(probs,k=min(TOPK,probs.numel()))
            gold=int(batch["user_labels"].item())
            rec["gold_user"]=idx2user_small[gold]
            rec["top15_user"]=[(idx2user_small[int(i)],float(v)) for i,v in zip(topi,topv)]
            metrics["user_mrr15"].append(mrr_at_k(topi.tolist(),gold))
            metrics["user_ndcg15"].append(ndcg_at_k(topi.tolist(),gold))

        if task==ITEM_TASK:
            probs=torch.softmax(out["item_logits"],dim=-1)[0]
            topv,topi=torch.topk(probs,k=min(TOPK,probs.numel()))
            gold=int(batch["item_labels"].item())
            rec["gold_item"]=idx2item[gold]
            rec["top15_item"]=[(idx2item[int(i)],float(v)) for i,v in zip(topi,topv)]
            metrics["item_mrr15"].append(mrr_at_k(topi.tolist(),gold))
            metrics["item_ndcg15"].append(ndcg_at_k(topi.tolist(),gold))

        if task in GEN_TASKS_WORD:
            lm_text=lm_generate_with_hints(out["word_logits"], batch["q_text"], max_new_tokens=60)
            rec["lm_pred_text"]=lm_text
            rec["gold_text"]=batch["t_text"]
            p=tokenize_simple(lm_text)
            r=tokenize_simple(batch["t_text"])
            metrics[f"{task}_rouge1"].append(rouge1_f1(p,r))
            metrics[f"{task}_rougeL"].append(rougeL_f1(p,r))

        records.append(rec)

results_df = pd.DataFrame(records)

print("\n===== USER =====")
print("MRR@15 :", mean_or_nan(metrics["user_mrr15"]))
print("NDCG@15:", mean_or_nan(metrics["user_ndcg15"]))

print("\n===== ITEM =====")
print("MRR@15 :", mean_or_nan(metrics["item_mrr15"]))
print("NDCG@15:", mean_or_nan(metrics["item_ndcg15"]))

for t in GEN_TASKS_WORD:
    print(f"\n===== {t} =====")
    print("ROUGE-1:", mean_or_nan(metrics[f"{t}_rouge1"]))
    print("ROUGE-L:", mean_or_nan(metrics[f"{t}_rougeL"]))

results_df.head()
